In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchaudio import transforms, datasets
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F

In [ ]:
train_data = datasets.SPEECHCOMMANDS(root='./data', download=True, subset='training')
test_data = datasets.SPEECHCOMMANDS(root='./data', download=True, subset='testing')
valid_data = datasets.SPEECHCOMMANDS(root='./data', download=True, subset='validation')



In [ ]:
label = list(set([i[2] for i in train_data]))
label

In [ ]:
len(label)

In [ ]:
label_to_index = {label: i for i, label in enumerate(label)}


In [ ]:
train = DataLoader(train_data, batch_size=64, shuffle=True)
test = DataLoader(test_data, batch_size=64)

In [ ]:
transform = transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=64
)

In [ ]:
max_len = 100

def collate_fn(batch):
  spectrograms, targets = [], []
  for waveform, sample_rate, label, *_ in batch:
    spec = transform(waveform).squeeze(0)

    if spec.shape[1] > max_len:
            spec = spec[:, :max_len]

    if spec.shape[1] < max_len:
            pad_amount = max_len - spec.shape[1]
            spec = F.pad(spec, (0, pad_amount))
    spectrograms.append(spec)
    targets.append(label_to_index[label])

  spectrograms = torch.stack(spectrograms)
  targets = torch.tensor(targets)
  return spectrograms, targets

In [ ]:


class AudioClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super(AudioClassifier, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((8, 8))
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
model = AudioClassifier(len(label)).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:

# Машыктыруу цикли
for epoch in range(5):
    model.train()  # Моделди машыктыруу режимине өткөрүү
    total_loss = 0

    for x_batch, y_batch in train:
        # Дайындарды GPU же CPU'га жиберүү
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # 1. Алдыга карай кадам (Forward pass)
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)

        # 2. Артка карай кадам (Backward pass)
        optimizer.zero_grad()  # Градиенттерди тазалоо
        loss.backward()        # Катанын градиентин эсептөө
        optimizer.step()       # Салмактарды (weights) жаңыртуу

        total_loss += loss.item()

    # Ар бир доордун (epoch) жыйынтыгын чыгаруу
    print(f"Эпоха {epoch+1}, Потери (Loss): {total_loss / len(train):.4f}")

In [ ]:
model.eval()  # Моделди баалоо режимине өткөрүү
correct, total = 0, 0

with torch.no_grad():  # Градиентти эсептөөнү өчүрүү (эстутумду үнөмдөйт)
    for x_batch, y_batch in test:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model(x_batch)

        # Эң жогорку ишенимдүүлүк коэффициенти бар классты тандоо
        predicted = torch.argmax(y_pred, dim=1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

# Тактыкты (accuracy) эсептөө
accuracy = 100 * correct / total
print(f"Точность модели на тестовых данных: {accuracy:.2f}%")

In [ ]:
torch.save(model.state_dict(), 'speech_commands_model.pth')

In [ ]:
torch.save(label, 'speechcommands_labels.pth')
